# §1.3 draft — source comparison and assumptions

Copy these cells into `wip_echo.ipynb` under `### 1.3 Source comparison and assumptions`.
Requires `json_tables`, `xml_tables` (§1.1–§1.2) and `pd`.

### 1.3 Source comparison and assumptions

**Why** Task 1 asks which fields appear in one or both sources, what the format conventions are,
where records duplicate within a source and overlap across sources, and which assumptions must
hold before transformation. Everything below is one of those four.

**What** Four findings and one assumptions register. No output tables are built here — that is
Jasmine's §4.

**How** Compare the two dicts of frames produced by §1.1 and §1.2, using a small set of
normalisers that exist only to make values comparable.

**So what** This section is the hand-off. Jasmine gets the format rules and the normalisers,
Yandu gets the duplicate and overlap counts his reconciliation must reproduce, Shawn gets
confirmation that the raw text fields agree across sources.

#### What §1 must show, and where it is shown

Spec §4 Task 1 lists seven things the notebook and the mapping must evidence. This table is the
index; every later cell in §1 discharges one of them. A reviewer should be able to start here.

| # | Spec requirement | Where | Evidence |
|---|---|---|---|
| 1 | major objects, nested arrays, repeated XML elements, source-specific representations | 1.1, 1.2 structure surveys | top-level keys and record counts; root children, one `Order`, the repeated `Shopping_Cart/Item` |
| 2 | candidate primary keys and **foreign keys** | 1.3g | key derived from the dictionary, candidate-key scan, FK closure tested per source and on the union |
| 3 | the grain of each source collection | 1.3g | dictionary `grain` column, confirmed by key uniqueness |
| 4 | date, timestamp, boolean, currency, percentage and missing-value formats | 1.3a, 1.3b | one order shown as each source writes it; format table; the normalisers that implement it |
| 5 | fields that appear in one or both sources | 1.3a | table-level and column-level set differences |
| 6 | duplicate records within a source, overlapping records across sources | 1.3c, 1.3d, 1.3e, 1.3f | duplicates proved field-identical; the row-count difference explained; overlap sized; negative control |
| 7 | assumptions needed before transformation | 1.3h | assumptions register, each with evidence and an owner |

The mapping CSV then carries the same findings per field — `|` separating multiple inputs, and
`notebook_evidence` citing these section names. The mapping describes the method; this notebook
is the method.

#### 1.3a Field coverage and format conventions

**Why** These are the differences Jasmine must reconcile, field by field. Stating them as a
table means she does not have to rediscover them from the raw files.

**What** Which tables exist in which source, then the same value shown as each source writes it.

**How** Set arithmetic on the table keys, then one shared order printed from both frames.

**So what** Every row of the format table becomes exactly one conversion in §4 and one
`VAL-SCHEMA-` check in §6. If a row here has no matching check there, something was missed.

In [ ]:
print("JSON tables:", sorted(json_tables))
print("XML  tables:", sorted(xml_tables))
print("JSON only  :", sorted(set(json_tables) - set(xml_tables)))
print("XML only   :", sorted(set(xml_tables) - set(json_tables)))

# Column coverage for the four tables both sources carry.
for name in sorted(set(json_tables) & set(xml_tables)):
    j, x = set(json_tables[name].columns), set(xml_tables[name].columns)
    print(f"\n{name}: {len(j & x)} shared columns")
    if j - x: print("   JSON only:", sorted(j - x))
    if x - j: print("   XML only :", sorted(x - j))

In [ ]:
# The same order, as each source writes it. This is the format evidence for A1.
sample_id = sorted(set(json_tables["orders"].order_id) & set(xml_tables["orders"].order_id))[0]
fields = ["order_timestamp", "order_price", "coupon_discount", "expedited_delivery", "coupon_code"]

side_by_side = pd.DataFrame({
    "JSON": json_tables["orders"].set_index("order_id").loc[sample_id, fields],
    "XML":  xml_tables["orders"].set_index("order_id").loc[sample_id, fields],
})
print("Order", sample_id)
side_by_side

The pattern, stated once so §4 can be written from it:

| Field type | JSON | XML | Rule for §4 |
|---|---|---|---|
| timestamp | `2018-12-31 13:52:00` (ISO) | `02/11/2018 08:50:00` (day first) | parse with `dayfirst=True` for XML only |
| date | `2019-01-08` | `10/11/2018` | same |
| money | `1189.23` (number) | `AUD 2,765.47` (string) | strip `AUD` and `,`, cast to float, round to 2dp |
| percent | `10` (number) | `10%` (string) | strip `%`, cast to float |
| boolean | `true` / `false` | `Y` / `N` | map to Python `bool` |
| missing string | `""` | empty element → `""` by DEC-015 | becomes the literal `"NaN"` at export, in §4 |

Every XML column arrives as text; every JSON column arrives natively typed. That single
sentence is the reason §1.3 exists.

#### 1.3b Normalisers

**Why** The two sources cannot be compared until the same value is written the same way. This is
also the moment the format rules above become code.

**What** Four functions, each one line of real work.

**How** `dayfirst` is passed in explicitly rather than guessed from the string — a heuristic that
sniffs for `/` would silently mis-parse `03/11/2018` the day a source changes.

**So what** These are the *canonical* implementations. Jasmine imports them into §4 rather than
writing her own; two implementations of "what is a valid date" is how C1 and E1 get lost. If
they move into `Group001_text_functions.py` or a shared cell, they move as a set.

In [ ]:
def norm_money(value):
    """'AUD 2,765.47' or 2765.47 -> 2765.47"""
    return round(float(str(value).replace("AUD", "").replace(",", "").strip()), 2)

def norm_percent(value):
    """'10%' or 10 -> 10.0"""
    return float(str(value).replace("%", "").strip())

def norm_bool(value):
    """'Y' / 'N' / True / False -> bool"""
    return str(value).strip().lower() in {"y", "yes", "true"}

def norm_datetime(value, dayfirst):
    """Text date or timestamp -> pandas Timestamp. `dayfirst` is True for XML, False for JSON."""
    return pd.to_datetime(str(value).strip(), dayfirst=dayfirst, errors="coerce")

# Which normaliser applies to which column. Used by the comparison below and reusable in §4.
NORMALISERS = {
    "orders":      {"money":   ["order_price", "delivery_charges", "tax_amount", "order_total"],
                    "percent": ["coupon_discount"],
                    "bool":    ["expedited_delivery"],
                    "datetime":["order_timestamp"],
                    "number":  ["customer_lat", "customer_long"]},
    "order_items": {"money":   ["unit_price", "line_revenue"],
                    "number":  ["quantity"]},
    "deliveries":  {"money":   ["delivery_cost"],
                    "bool":    ["on_time_in_full", "signature_required"],
                    "datetime":["dispatch_date", "promised_date", "delivered_date"],
                    "number":  ["delay_days", "fulfilment_hours", "promised_days",
                                "tracking_event_count", "shipping_distance_km",
                                "estimated_carbon_kg"]},
    "product_reviews": {"bool":    ["verified_purchase"],
                        "datetime":["review_timestamp"],
                        "number":  ["rating", "helpful_votes"]},
}

def normalise(series, column, spec, dayfirst):
    """Apply the right normaliser to one column. Anything unlisted is compared as text."""
    if column in spec.get("money", []):     return series.map(norm_money)
    if column in spec.get("percent", []):   return series.map(norm_percent)
    if column in spec.get("bool", []):      return series.map(norm_bool)
    if column in spec.get("datetime", []):  return series.map(lambda v: norm_datetime(v, dayfirst))
    if column in spec.get("number", []):    return series.astype(float)
    return series.astype(str)

#### 1.3c Duplicates within each source

**Why** Two questions decide how hard reconciliation is: how many duplicate business keys are
there, and are the duplicate rows identical? If they disagree, someone has to choose a winner.

**What** Per source and table: duplicated key rows, and whether the full rows are field-identical.

**How** Count rows sharing a key, then count distinct full rows among them. Equal counts means
every duplicate is an exact copy of its twin.

**So what** This is Yandu's Q2. If everything is field-identical, the canonical-row rule is
arbitrary and only needs to be deterministic and documented. If not, it needs a real precedence
argument. **Run this before any `drop_duplicates()` anywhere** — deduplicating first and asking
afterwards assumes the answer.

In [ ]:
# The intended key of each table, read from the dictionary rather than assumed:
# every table lists its identifier at position 1, and `grain` says so in words.
DICT = pd.read_csv(DICT_PATH, keep_default_na=False)
KEYS = DICT[DICT.position == 1].set_index("output_table")["field_name"].to_dict()
print(DICT[DICT.position == 1][["output_table", "field_name", "grain"]].to_string(index=False))

rows = []
for source, tables in [("JSON", json_tables), ("XML", xml_tables)]:
    for name, key in KEYS.items():
        if name not in tables:
            continue
        df = tables[name]
        dupes = df[df[key].duplicated(keep=False)]
        distinct_rows = dupes.drop(columns="source_system").astype(str).drop_duplicates()
        rows.append({
            "source": source, "table": name,
            "rows": len(df), "unique_keys": df[key].nunique(),
            "duplicate_key_rows": len(dupes),
            "duplicated_keys": dupes[key].nunique(),
            "distinct_full_rows": len(distinct_rows),
            "all_identical": len(distinct_rows) == dupes[key].nunique(),
        })

pd.DataFrame(rows)

**Observed.** Every duplicated key resolves to exactly one distinct full row, in both sources and
every table. No duplicate pair disagrees on any field.

**Interpretation for Q2.** The canonical-row choice is genuinely arbitrary — `drop_duplicates`
keeping the first occurrence loses nothing. Yandu still has to state the rule and show it is
deterministic, but he does not need a precedence argument. This cell is his evidence.

#### 1.3d The `order_items` row-count difference

**Why** JSON gives 8,826 item rows and XML 8,833. Left unexplained, that looks like a parser bug,
and it is the first thing a reviewer will ask about.

**What** Whether the seven extra rows are extra *items* or extra *copies*.

**How** Separate unique item ids from duplicated rows, then check whether the two sources
duplicate the same orders.

**So what** If it were extra items, `parse_xml` would be flattening something wrong and Jasmine
must not build on it. The answer decides whether §1.2 ships.

In [ ]:
for source, tables in [("JSON", json_tables), ("XML", xml_tables)]:
    items, orders = tables["order_items"], tables["orders"]
    duplicated_orders = set(orders.order_id[orders.order_id.duplicated()])
    print(f"{source}: {len(items):,} item rows = {items.order_item_id.nunique():,} unique "
          f"+ {int(items.order_item_id.duplicated().sum()):,} duplicate rows")
    print(f"        {len(duplicated_orders)} duplicated orders carry "
          f"{len(items[items.order_id.isin(duplicated_orders)]):,} of those rows")

dup_json = set(json_tables["orders"].order_id[json_tables["orders"].order_id.duplicated()])
dup_xml  = set(xml_tables["orders"].order_id[xml_tables["orders"].order_id.duplicated()])
print(f"\nDuplicated orders: JSON {len(dup_json)}, XML {len(dup_xml)}, "
      f"appearing in both: {len(dup_json & dup_xml)}")

**Observed.** Every extra row is a duplicate, not a new item: 8,625 unique + 201 duplicates in
JSON, 8,619 unique + 214 in XML. The 68 duplicated orders in each file share **zero** order ids.

**Interpretation.** The two files duplicate *different* orders, and those orders happen to carry
different numbers of cart lines — 201 duplicated item rows against 214. That fully accounts for
the seven-row difference. Nothing is being flattened incorrectly, and `parse_xml` is sound.

This also matters for §5: because the duplicated sets are disjoint, deduplicating within each
source before the union gives the same answer as deduplicating after it. Yandu should not rely on
that holding — it is a property of this data, not a rule — but it is a useful cross-check.

#### 1.3e Overlap across the two sources

**Why** The spec says records may overlap across sources. Whether the overlapping copies *agree*
decides whether a source-precedence rule is needed at all.

**What** Per table: how many keys are in each source, in both, and in the union; then whether any
field disagrees on the shared keys after normalisation.

**How** Deduplicate first — safe only because 1.3c proved duplicates are identical — then compare
column by column with the normalisers from 1.3b.

**So what** A precedence rule that is never exercised is dead code a marker will ask about; no
rule at all is a gap. This cell decides which of those two risks the group is taking.

In [ ]:
overlap = []
for name in ["orders", "order_items", "deliveries", "product_reviews"]:
    key = KEYS[name]
    j, x = set(json_tables[name][key]), set(xml_tables[name][key])
    overlap.append({"table": name, "json_keys": len(j), "xml_keys": len(x),
                    "in_both": len(j & x), "union": len(j | x)})

pd.DataFrame(overlap)

In [ ]:
def compare_shared(name):
    """Field-by-field comparison of the rows both sources carry. Returns {column: n_disagreements}."""
    key  = KEYS[name]
    spec = NORMALISERS.get(name, {})
    shared = sorted(set(json_tables[name][key]) & set(xml_tables[name][key]))

    left  = json_tables[name].drop_duplicates(key).set_index(key).loc[shared]
    right = xml_tables[name].drop_duplicates(key).set_index(key).loc[shared]

    disagreements = {}
    for column in left.columns:
        if column == "source_system":
            continue
        a = normalise(left[column],  column, spec, dayfirst=False)   # JSON: ISO
        b = normalise(right[column], column, spec, dayfirst=True)    # XML: day first
        # NaT != NaT and NaN != NaN are both True in pandas, so two equally-missing
        # values would count as a disagreement. Exclude that case explicitly.
        n = int(((a != b) & ~(a.isna() & b.isna())).sum())
        if n:
            disagreements[column] = n
    return len(shared), disagreements


for name in ["orders", "order_items", "deliveries", "product_reviews"]:
    n_shared, disagreements = compare_shared(name)
    print(f"{name:16s} {n_shared:>6,} shared keys -> {disagreements or 'no disagreements'}")

#### 1.3f Negative control

**Why** The cell above reports zero disagreements everywhere. A check that has never been seen to
fail is indistinguishable from a check that cannot fail, and that is exactly the objection a
marker raises against a clean result.

**What** One deliberately corrupted value, run through the same comparison.

**How** Copy one shared order, change one field, re-compare. Nothing outside this cell is
touched — the corrupted frame is local and discarded.

**So what** This is the evidence that "no disagreements" is a finding rather than a broken test.
It is also the template for Yandu's `VAL-` checks: every check he writes should be able to show
what it looks like when it fails.

In [ ]:
# Corrupt one field in a copy of the XML orders, then re-run the same comparison.
key = "orders"
broken = xml_tables[key].copy()
victim = sorted(set(json_tables[key].order_id) & set(broken.order_id))[0]
broken.loc[broken.order_id == victim, "order_price"] = "AUD 1.00"

saved = xml_tables[key]
try:
    xml_tables[key] = broken                         # swap in the corrupted frame
    n_shared, disagreements = compare_shared(key)
finally:
    xml_tables[key] = saved                          # always swap it straight back

print(f"With one corrupted price on order {victim}: {disagreements}")
print("Restored cleanly:", compare_shared(key)[1] or "no disagreements")

#### 1.3g Keys, grain and referential integrity

**Why** Spec bullets 2 and 3: candidate primary keys, foreign keys, and the grain of each
collection. Grain and key are the same statement made two ways — "one row per order" is a claim
that `order_id` is unique.

**What** Three steps: the intended key from the dictionary, every column that is *actually*
unique in the data, and whether each foreign key resolves.

**How** Uniqueness is tested on de-duplicated rows, since 1.3c showed duplicates are exact copies.
Foreign keys are tested twice — inside each source alone, then on a de-duplicated union.

**So what** The union test is the one that matters, and its result changes what Jasmine and Yandu
are allowed to do. Read the output before writing any join.

In [ ]:
def candidate_keys(df):
    """Columns unique across de-duplicated rows.
    Necessary for a primary key, NOT sufficient — uniqueness here can be coincidence."""
    base = df.drop(columns="source_system").astype(str).drop_duplicates()
    return [c for c in base.columns if base[c].nunique() == len(base)]

for source, tables in [("JSON", json_tables), ("XML", xml_tables)]:
    for name, df in tables.items():
        if name not in KEYS:                 # warehouses is not an output table
            continue
        found = candidate_keys(df)
        others = [c for c in found if c != KEYS[name]]
        status = "OK" if KEYS[name] in found else "** NOT UNIQUE **"
        print(f"{source} {name:16s} key {KEYS[name]:14s} {status:16s} also unique: {others or 'none'}")

**Observed.** The dictionary key is unique in every table and both sources. Four tables carry a
second unique column:

| Table | Also unique | Is it a key? |
|---|---|---|
| `orders` | `source_system_record_id` | **Yes — alternate key.** An independent handle on the same row; useful to Yandu as a second way to match overlapping orders. |
| `deliveries` | `order_id` | **Yes — and it is the evidence for the grain.** One row per order, so the FK to `orders` is 1:1, not many-to-one. |
| `products` | `product_name`, `product_sku` | `product_sku` yes; `product_name` is coincidence. Shawn's `extract_product_sku` output joins on the SKU. |
| `product_reviews` | `order_item_id`, `review_body_raw` | `order_item_id` yes — at most one review per order item. `review_body_raw` is coincidence: 3,850 long strings that happen not to repeat. |

`customers.lifetime_value_before_period` is unique across all 500 rows and is obviously not a
key. That is the point of the "necessary, not sufficient" line above — a key is chosen from the
dictionary and *confirmed* by uniqueness, never selected by it.

In [ ]:
# Foreign keys implied by the data dictionary's field names and grains.
FOREIGN_KEYS = [
    ("order_items",     "order_id",      "orders",      "order_id"),
    ("order_items",     "product_id",    "products",    "product_id"),
    ("orders",          "customer_id",   "customers",   "customer_id"),
    ("deliveries",      "order_id",      "orders",      "order_id"),
    ("product_reviews", "order_id",      "orders",      "order_id"),
    ("product_reviews", "order_item_id", "order_items", "order_item_id"),
    ("product_reviews", "product_id",    "products",    "product_id"),
    ("product_reviews", "customer_id",   "customers",   "customer_id"),
]

def check_foreign_keys(label, tables):
    print(f"--- {label} ---")
    for child, child_col, parent, parent_col in FOREIGN_KEYS:
        if child not in tables:
            continue
        values = set(tables[child][child_col])
        if parent not in tables:
            print(f"  {child}.{child_col:14s} -> {parent:16s} parent table absent from this source")
            continue
        orphans = values - set(tables[parent][parent_col])
        print(f"  {child}.{child_col:14s} -> {parent:16s} {len(values):>6,} distinct, {len(orphans):>5,} orphans")

check_foreign_keys("JSON alone", json_tables)
check_foreign_keys("XML alone", xml_tables)

In [ ]:
# A de-duplicated union, built here ONLY to test referential integrity.
# The reconciled tables themselves are produced in §5 — this is throwaway.
union = {}
for name in set(json_tables) | set(xml_tables):
    parts = [t[name] for t in (json_tables, xml_tables) if name in t]
    combined = pd.concat(parts, ignore_index=True)
    union[name] = combined.drop_duplicates(KEYS[name]) if name in KEYS else combined

check_foreign_keys("UNION of both sources", union)
print()
for name in sorted(KEYS):
    if name in union:
        print(f"union {name:16s} {len(union[name]):>6,} rows")

**Observed — this is the most important finding in §1.**

Inside a single source, referential integrity **fails**. `product_reviews` has 1,300 orphan
`order_id` values in JSON and 1,278 in XML, plus roughly 1,800 orphan `order_item_id` values in
each. `products` is missing entirely from JSON and `customers` entirely from XML, so those
foreign keys cannot even be evaluated per source.

On the de-duplicated union, **every foreign key resolves with zero orphans**.

**Interpretation.** The two exports are not two copies of the same data with a little overlap.
They are *complementary slices*: a review can sit in the JSON file while the order it points at
sits only in the XML file. Three consequences, and each belongs to someone:

- **Jasmine (§4)** — never filter a child table to rows whose parent exists in the same source.
  Doing that to `product_reviews` before integration would silently delete about a third of them.
- **Yandu (§6)** — every `VAL-FK-` check runs on the integrated tables, never per source. A
  per-source FK check reports thousands of failures that are not failures.
- **Everyone** — this is the strongest available evidence that the parsing and union are correct.
  Two independently parsed files, unioned and de-duplicated, produce exact 1:1 closure on eight
  foreign keys at once. A flattening error almost anywhere would break it.

#### 1.3h Assumptions register

The assumptions the rest of the workflow depends on. Each is either evidenced above or flagged as
unverified, and each names who breaks if it is wrong.

| # | Assumption | Evidence | Who depends on it |
|---|---|---|---|
| A-01 | The business key of each table is the `*_id` field in the data dictionary | 1.3c — every non-duplicate key is unique | Jasmine (PK), Yandu (`VAL-PK-`) |
| A-02 | Duplicate rows within a source are exact copies, so the canonical-row choice is arbitrary | 1.3c — distinct full rows equal duplicated keys, all tables, both sources | Yandu (Q2) |
| A-03 | Overlapping records agree on every field after normalisation, so no source-precedence rule is needed | 1.3e, with 1.3f as the control | Yandu (C1, C2) |
| A-04 | Conflict *detection* is still required even though no conflict exists | spec §4 asks for the rule, not for a conflict | Yandu (C2) |
| A-05 | XML values are text and JSON values are natively typed; the format table in 1.3a is complete | 1.3a | Jasmine (§4), Yandu (`VAL-SCHEMA-`) |
| A-06 | An empty XML element and an empty JSON string mean the same thing (DEC-015) | 1.3e — on the 500 shared orders, `coupon_code` shows no disagreement, so `""` in one source matches `""` in the other. Whole-file blank counts (1,767 JSON vs 1,770 XML) are *not* evidence: the two files hold different order populations. | everyone |
| A-07 | `warehouses` maps to no output field and is analysis-only | data dictionary has no warehouse table | Jasmine (must not export it) |
| A-08 | Raw text fields are byte-identical across sources for overlapping records | 1.3e — `customer_note_raw` and `review_body_raw` show no disagreement | Shawn (one cleaning path, not two) |
| A-09 | Referential integrity holds **only on the union**; per-source orphans are expected and are not errors | 1.3g — 8/8 foreign keys close with zero orphans after union, none do per source | Jasmine (§4 joins), Yandu (`VAL-FK-`) |
| A-10 | The two exports are complementary slices of one dataset, not duplicate copies | 1.3g, plus 1.3e — only 500 of 5,000 orders overlap | everyone |
| A-11 | A column being unique in this data does not make it a key | 1.3g — `lifetime_value_before_period`, `review_body_raw`, `product_name` are unique by coincidence | Yandu (`VAL-PK-`) |
| A-12 | **Unverified:** that arithmetic (line revenue → order price → tax → total) is internally consistent | not checked here — it is `VAL-ARITH-` in §6 | Yandu |

#### Hand-off

- **Jasmine** — the format table in 1.3a and the normalisers in 1.3b. Import them; do not rewrite.
- **Yandu** — 1.3c answers Q2, 1.3e sizes the overlap, 1.3f is the pattern for every `VAL-` check.
- **Shawn** — A-08: the raw text is identical across sources, so there is one cleaning path.
  Extraction still runs on `_raw` before cleaning (DEC-014).

Open: whether the 1.3b normalisers live here or move to a shared module for §4. They are
currently defined in Echo's notebook, which means Jasmine cannot import them without copying.